In [2]:
import openai

print(openai.__version__)

2.14.0


In [3]:
import os
import sys

os.chdir("..")
sys.path.append("src/")

In [3]:
from openai import OpenAI

model = "gpt-5.1"
client = OpenAI()

user_prompt = "Hi, my name is Maliheh"
response = client.responses.create(model=model, input=user_prompt)

print(response.output_text)

Nice to meet you, Maliheh. How can I help you today?


In [4]:
from pydantic import BaseModel


class EventExtractInfo(BaseModel):
    name: str
    date: str
    participants: list[str]


response = client.responses.parse(
    model=model,
    input=[
        {"role": "system", "content": "Extract the event information"},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=EventExtractInfo,
)

print(response.output_parsed)

name='science fair' date='Friday' participants=['Alice', 'Bob']


In [5]:
model = "gpt-4.1-mini"

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "what's in this image?"},
                {
                    "type": "input_image",
                    "image_url": "https://www.eatingwell.com/thmb/8MXOea6bVolkuLNQ1HhNt1tryIE=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/article_7866255_foods-you-should-eat-every-week-to-lose-weight_-04-d58e9c481bce4a29b47295baade4072d.jpg",
                },
            ],
        }
    ],
)

print(response.output_text)

The image shows a variety of foods arranged on a light blue surface. Included are:

- A cut wedge of red cabbage
- Brown rice in a bowl
- Baby bok choy
- A fillet of raw salmon on a plate
- Broccoli rabe or broccolini
- An avocado, one half with seed and one whole
- Two bars of dark chocolate
- Three eggs
- A bowl of sauerkraut
- A bowl of unpeeled pistachios
- A small bowl of chia seeds
- A bowl of wheat or some type of grain
- Two red apples
- A bowl of kimchi

These ingredients represent a mix of vegetables, fruits, grains, protein sources, and fermented foods.


In [6]:
import base64

model = "gpt-4.1"


def encode_image_by_path(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_path = (
    "/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp"
)
base64_image = encode_image_by_path(image_path)

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "what's in this image?"},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
)

print(response.output_text)

This image contains a variety of healthy foods, including:

- Salmon fillet
- Bok choy (baby)
- Red cabbage (halved)
- Broccolini
- Eggs
- Brown rice (uncooked)
- Whole grain or farro (uncooked)
- Apples
- Sauerkraut
- Kimchi
- Dark chocolate
- Pistachios
- Chia seeds
- Avocado

These foods are nutrient-dense and commonly associated with healthy eating patterns.


In [7]:
# lets you convert files (like images) into Base64 text


def encode_image_by_path(image_path):
    # Step 1: Open the image as bytes (binary data)
    # rb: read binary (images are not text, they're raw bytes)
    image_file = open(image_path, "rb")

    # Step 2: Read all the bytes from the image
    image_bytes = image_file.read()

    # Step 3: Convert those bytes into Base64 bytes
    base64_bytes = base64.b64encode(image_bytes)

    # Step 4: Convert Base64 bytes into a normal string
    base64_string = base64_bytes.decode("utf-8")

    # Step 5: Return the final Base64 string
    return base64_string

In [8]:
raw_data = b"\x89PNG\r\n\x1a\n"  # This is how real image bytes often look
print("RAW BYTES:", raw_data)

# Step 1: Base64 encode the raw bytes
base64_bytes = base64.b64encode(raw_data)
print("BASE64 (bytes):", base64_bytes)

# Step 2: Convert Base64 bytes → string
base64_string = base64_bytes.decode("utf-8")
print("BASE64 (string):", base64_string)

RAW BYTES: b'\x89PNG\r\n\x1a\n'
BASE64 (bytes): b'iVBORw0KGgo='
BASE64 (string): iVBORw0KGgo=


In [9]:
from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_path
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_path = (
    "/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp"
)
base64_image = encode_image_by_path(image_path)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format=IngredientsResponse,
)

print(response)

OPIK: Started logging traces to the "food-image-analyzer" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019bcb69-a5fe-7d06-bb0d-a4b8d7182a0f&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBp.


name='Healthy Foods Platter' ingredients=[Ingredient(ingredient_name='Red cabbage', portiont='1/4 head'), Ingredient(ingredient_name='Baby bok choy', portiont='3 small'), Ingredient(ingredient_name='Salmon fillet', portiont='1 fillet (~120g)'), Ingredient(ingredient_name='Broccolini', portiont='3 stalks'), Ingredient(ingredient_name='Brown rice', portiont='1/2 cup (uncooked)'), Ingredient(ingredient_name='Farro (whole grain)', portiont='1/2 cup (uncooked)'), Ingredient(ingredient_name='Eggs', portiont='3 whole'), Ingredient(ingredient_name='Sauerkraut', portiont='1/2 cup'), Ingredient(ingredient_name='Avocado', portiont='1 whole, halved'), Ingredient(ingredient_name='Dark chocolate', portiont='5 pieces (about 30g)'), Ingredient(ingredient_name='Pistachios', portiont='1 handful (about 1/4 cup, shelled)'), Ingredient(ingredient_name='Apples', portiont='2 whole'), Ingredient(ingredient_name='Kimchi', portiont='1/2 cup'), Ingredient(ingredient_name='Chia seeds', portiont='2 tablespoons')]


In [10]:
from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_image_url(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    image_url=image_url,
    response_format=IngredientsResponse,
)

print(response)

name='Vietnamese noodle soup (possibly Hu Tieu or Bun Rieu)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1-2 cups, cooked'), Ingredient(ingredient_name='Shrimp', portiont='2 whole pieces'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Sliced pork', portiont='3-4 pieces'), Ingredient(ingredient_name='Broth (possibly pork or seafood)', portiont='1-2 cups'), Ingredient(ingredient_name='Fresh herbs (mint, possibly cilantro)', portiont='A few sprigs'), Ingredient(ingredient_name='Scallions/green onions', portiont='1 tablespoon, chopped'), Ingredient(ingredient_name='Red chili peppers', portiont='1 tablespoon, sliced'), Ingredient(ingredient_name='Fried shallots/garlic (optional, visible bits)', portiont='1 teaspoon')]


In [11]:
from pydantic import BaseModel

from services.analysis.schemas import IngredientsResponse
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_url
from services.prompts import FoodImageAnalyzerPrompts

model = "gpt-4.1"
image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"
base64_image = encode_image_by_url(image_url)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model,
    system_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=FoodImageAnalyzerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format=IngredientsResponse,
)

print(response)

name='Vietnamese Noodle Soup (Hu Tieu)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 bowl'), Ingredient(ingredient_name='Shrimp', portiont='2 pieces'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Pork slices', portiont='4-5 slices'), Ingredient(ingredient_name='Chopped chili peppers', portiont='1 tablespoon'), Ingredient(ingredient_name='Chopped green onions', portiont='1 tablespoon'), Ingredient(ingredient_name='Fresh herbs (cilantro, mint)', portiont='Several leaves'), Ingredient(ingredient_name='Broth', portiont='1.5 cups')]


In [12]:
from services.analysis.ingredients import IngredientsAnalyzer

image_url = "https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg"
ingredients_analyzer = IngredientsAnalyzer()
result = ingredients_analyzer.analyze(image_url)
print(result)

2026-01-17 11:04:39.687 | INFO     | services.cache.client:get:18 - Cached result found for key ingredients:db833fcd3a2d9c0cd22f58deafe32bd94b880d2169ee0485e284b8792da9520e


name='Vietnamese Hu Tieu (Seafood & Pork Noodle Soup)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 cup'), Ingredient(ingredient_name='Shrimp', portiont='2 large pieces'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Pork slices', portiont='5-6 thin slices'), Ingredient(ingredient_name='Fresh herbs (cilantro, mint, etc.)', portiont='2 sprigs'), Ingredient(ingredient_name='Sliced red chili', portiont='1 tablespoon'), Ingredient(ingredient_name='Scallions (green onions)', portiont='1 tablespoon, chopped'), Ingredient(ingredient_name='Broth (savory soup base)', portiont='1.5 cups')]


In [13]:
from services.analysis.nutrients import NutrientsAnalyzer
from services.analysis.schemas import Ingredient

nutrients_analyzer = NutrientsAnalyzer()

ingredients = [
    Ingredient(ingredient_name="Rice noodles", portiont="1 serving"),
    Ingredient(ingredient_name="Shrimp", portiont="2 pieces"),
    Ingredient(ingredient_name="Quail eggs", portiont="4 pieces"),
    Ingredient(ingredient_name="Pork slices", portiont="4-5 slices"),
    Ingredient(ingredient_name="Fresh herbs (such as mint)", portiont="A handful"),
    Ingredient(ingredient_name="Green onions", portiont="1 tablespoon, chopped"),
    Ingredient(ingredient_name="Red chili pepper", portiont="1 tablespoon, sliced"),
    Ingredient(ingredient_name="Broth (pork or seafood-based)", portiont="1 cup"),
]
result = nutrients_analyzer.analyze(ingredients)
print(result)

2026-01-17 11:04:39.728 | INFO     | services.cache.client:get:18 - Cached result found for key nutrients:Rice noodles:1 serving_Shrimp:2 pieces_Quail eggs:4 pieces_Pork slices:4-5 slices_Fresh herbs (such as mint):A handful_Green onions:1 tablespoon, chopped_Red chili pepper:1 tablespoon, sliced_Broth (pork or seafood-based):1 cup
2026-01-17 11:04:40.816 | INFO     | services.cache.client:set:26 - Setting result for key nutrients:Rice noodles:1 serving_Shrimp:2 pieces_Quail eggs:4 pieces_Pork slices:4-5 slices_Fresh herbs (such as mint):A handful_Green onions:1 tablespoon, chopped_Red chili pepper:1 tablespoon, sliced_Broth (pork or seafood-based):1 cup


total_calories=390 total_protein_g=22.0 total_carbohydrates_g=49.0 total_fats_g=10.0 total_fiber_g=2.0


# Evaluation

In [4]:
from opik import Opik
client = Opik()
data_set = client.get_dataset('Nutrition-5K')
print(len(data_set.get_items()))
print(data_set.get_items()[0])

8
{'id': '019bc7c7-308a-7288-b498-ef88b4d502f5', 'carbohydrates': 18.445999, 'dish_id': 'dish_1568401064', 'img_url': 'https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/dish_1568401064/rgb.png', 'total_calories': 787.709961, 'mass': 572, 'protein': 108.667999, 'fat': 31.225998, 'ingredients': "['broccoli', 'mixed greens', 'chicken', 'avocado']", 'ingredients_mass': "[{'broccoli': 71.0}, {'mixed greens': 42.0}, {'chicken': 334.0}, {'avocado': 125.0}]"}


### Step 1: Define a evaluation_task function that a dataset item and runs the service and returns llm output and expected output

In [14]:
from services.analysis.ingredients import IngredientsAnalyzer
from services.analysis.nutrients import NutrientsAnalyzer

def evaluation_task(data_set_item):
    ingredients_analyzer = IngredientsAnalyzer()
    ingredients_result = ingredients_analyzer.analyze(data_set_item['img_url'])

    nutrients_analyzer = NutrientsAnalyzer()
    nutrients_result = nutrients_analyzer.analyze(ingredients_result.ingredients)

    predicted_output = {
        "ingredients": [ingredient.ingredient_name for ingredient in ingredients_result.ingredients],
        "nutrients": {
            'carbohydrates': nutrients_result.total_carbohydrates_g,
            'protein': nutrients_result.total_protein_g, 
            'fat': nutrients_result.total_fats_g,
            'total_calories': nutrients_result.total_calories,
        },
    }

    expected_output = {
        "ingredients": data_set_item['ingredients'],
        "nutrients": {
            'carbohydrates': data_set_item['carbohydrates'],
            'protein': data_set_item['protein'], 
            'fat': data_set_item['fat'],
            'total_calories': data_set_item['total_calories'],
        },
    }

    return {"predicted_output": predicted_output, "expected_output": expected_output}


print(evaluation_task(data_set.get_items()[0]))

2026-01-18 10:05:50.843 | INFO     | services.cache.client:get:18 - Cached result found for key ingredients:cd971f5c36219e22ed2cc350b9731cdb38ae13c23f7eb2d1c822014e96543c0d
2026-01-18 10:05:50.866 | INFO     | services.cache.client:get:18 - Cached result found for key nutrients:Grilled chicken breast:about 1 cup, diced_Avocado:1 small avocado, sliced_Mixed greens (lettuce, spinach, etc.):about 1 cup_Broccoli florets:about 1/2 cup


{'predicted_output': {'ingredients': ['Grilled chicken breast', 'Avocado', 'Mixed greens (lettuce, spinach, etc.)', 'Broccoli florets'], 'nutrients': {'carbohydrates': 22.0, 'protein': 41.0, 'fat': 21.0, 'total_calories': 430.0}}, 'expected_output': {'ingredients': "['broccoli', 'mixed greens', 'chicken', 'avocado']", 'nutrients': {'carbohydrates': 18.445999, 'protein': 108.667999, 'fat': 31.225998, 'total_calories': 787.709961}}}


### Step 2: Define a score function that gets llm and expected outputs and returns scores for one dataset item


In [17]:
import math
from pydantic import BaseModel

from services.chat_gpt.gpt import ChatGPT
from services.prompts import FoodImageAnalyzerPrompts


class ConfusionMatrix(BaseModel):
    true_positive: int
    false_positive: int
    false_negative: int

def score_ingredients(predicted_output, expected_output):
    chat_gpt = ChatGPT()
    predicted_ingredients = predicted_output['ingredients']
    expected_ingredients = expected_output['ingredients']
    ingredients_response = chat_gpt.generate_parsed_response(
        system_prompt=FoodImageAnalyzerPrompts.SCORE_INGREDIENTS_SYSTEM_PROMPT,
        user_prompt=FoodImageAnalyzerPrompts.SCORE_INGREDIENTS_USER_PROMPT.format(ingredients_list=predicted_ingredients, expected_ingredients_list=expected_ingredients, score_response_format=ConfusionMatrix),
        response_format=ConfusionMatrix
    )
    return ingredients_response

def score_nutrients(predicted_output, expected_output):
    predicted_nutrients = predicted_output["nutrients"]
    expected_nutrients = expected_output["nutrients"]

    nutrient_keys = ["carbohydrates", "protein", "fat", "total_calories"]

    absolute_errors = []
    squared_errors = []

    for key in nutrient_keys:
        error = predicted_nutrients[key] - expected_nutrients[key]

        absolute_errors.append(abs(error))
        squared_errors.append(error ** 2)

    mae = sum(absolute_errors) / len(absolute_errors)
    rmse = math.sqrt(sum(squared_errors) / len(squared_errors))

    return {
        "mae": mae,
        "rmse": rmse,
    }

data_set_item = data_set.get_items()[0]
evaluation_response = evaluation_task(data_set_item)
predicted_output, expected_output = (
    evaluation_response["predicted_output"],
    evaluation_response["expected_output"],
)
score_ingredients_response = score_ingredients(predicted_output, expected_output)
score_nutrients_response = score_nutrients(predicted_output, expected_output)

print(score_ingredients_response)
print(score_nutrients_response)

2026-01-18 10:07:32.916 | INFO     | services.cache.client:get:18 - Cached result found for key ingredients:cd971f5c36219e22ed2cc350b9731cdb38ae13c23f7eb2d1c822014e96543c0d
2026-01-18 10:07:32.939 | INFO     | services.cache.client:get:18 - Cached result found for key nutrients:Grilled chicken breast:about 1 cup, diced_Avocado:1 small avocado, sliced_Mixed greens (lettuce, spinach, etc.):about 1 cup_Broccoli florets:about 1/2 cup


true_positive=4 false_positive=0 false_negative=0
{'mae': 109.78948975, 'rmse': 182.10750687814158}


In [ ]:
from opik import Opik
client = Opik()

def score_ingredients_and_nutrients():
    data_set = client.get_dataset('Nutrition-5K')
    for data_set_item in data_set.get_items():
        evaluation_response = evaluation_task(data_set_item)
        predicted_output, expected_output = (
            evaluation_response["predicted_output"],
            evaluation_response["expected_output"],
        )
        score_ingredients_response = score_ingredients(predicted_output, expected_output)
        score_nutrients_response = score_nutrients(predicted_output, expected_output)

